<a href="https://colab.research.google.com/github/suhrudsharma/STP-Performance-Analysis/blob/main/%20notebooks/STP_DataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Spanning Tree Protocol (STP) Performance Analysis

## Notebook 1: Experimental Data Analysis

### Objective
Analyze experimental measurements collected from STP network topologies and link-failure scenarios.

### Tasks
1. Import experimental data from CSV files.
2. Validate and clean the data.
3. Calculate packet loss and convergence metrics.
4. Summarize results by topology and failure scenario.
5. Compare experimental performance across topologies.
6. Export cleaned data and summary tables for visualization and reporting.

### Data integrity
All experimental measurements must originate from actual network experiments. Missing measurements must not be fabricated. Calculated values must be distinguishable from raw measurements.

### Team
Swetha, Suhrud, and Isha


In [1]:

# Cell 1: Import libraries and verify the environment

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("STP Performance Analysis")
print("------------------------")
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Matplotlib version:", plt.matplotlib.__version__)

# Create output folders in the current Colab session
os.makedirs("data/processed", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/graphs", exist_ok=True)

print("\nRequired libraries imported successfully.")
print("Output folders created successfully.")


STP Performance Analysis
------------------------
Python version: 3.13.18
Pandas version: 2.2.3
NumPy version: 2.1.3
Matplotlib version: 3.10.0

Required libraries imported successfully.
Output folders created successfully.



## 2. Experimental Dataset Schema

This section defines the common data format used by the STP experiments.

Each row represents one experimental trial for a particular network topology and failure scenario.

Raw observations will be supplied by the network-emulation experiments. Derived metrics, such as packet-loss percentage and summary statistics, will be calculated programmatically.

Missing experimental values must remain missing rather than being replaced with invented measurements.


In [2]:

# Cell 2: Define the shared experimental data schema

REQUIRED_COLUMNS = [
    "experiment_id",
    "topology",
    "trial_number",
    "switch_count",
    "failed_link",
    "root_bridge",
    "blocked_ports",
    "forwarding_ports",
    "path_cost",
    "packets_sent",
    "packets_received",
    "packets_lost",
    "convergence_time_ms",
    "downtime_ms",
    "bpdu_count",
    "data_source",
    "notes",
]

NUMERIC_COLUMNS = [
    "trial_number",
    "switch_count",
    "blocked_ports",
    "forwarding_ports",
    "path_cost",
    "packets_sent",
    "packets_received",
    "packets_lost",
    "convergence_time_ms",
    "downtime_ms",
    "bpdu_count",
]

print("Dataset schema defined successfully.")
print("Total columns:", len(REQUIRED_COLUMNS))
print("\nColumns:")
for column in REQUIRED_COLUMNS:
    print("-", column)


Dataset schema defined successfully.
Total columns: 17

Columns:
- experiment_id
- topology
- trial_number
- switch_count
- failed_link
- root_bridge
- blocked_ports
- forwarding_ports
- path_cost
- packets_sent
- packets_received
- packets_lost
- convergence_time_ms
- downtime_ms
- bpdu_count
- data_source
- notes



## 3. Load Experimental Data

This section provides a reusable function for loading STP experimental data from a CSV file.

The function checks whether the expected columns are present. Actual experimental measurements will be loaded when the network experiments are completed.


In [5]:

# Cell 3: Reusable CSV loader

from pathlib import Path

def load_stp_data(file_path):
    """
    Load an STP experimental dataset from CSV.

    Parameters
    ----------
    file_path : str or Path
        Path to the CSV file.

    Returns
    -------
    pandas.DataFrame
        Loaded dataset after basic schema validation.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"CSV file not found: {file_path}\n"
            "Check the path or upload the experimental CSV."
        )

    df = pd.read_csv(file_path)

    missing_columns = [
        col for col in REQUIRED_COLUMNS
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "The CSV is missing required columns: "
            + ", ".join(missing_columns)
        )

    print("CSV loaded successfully.")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    return df


print("STP CSV loader defined successfully.")
print("Expected columns:", len(REQUIRED_COLUMNS))


STP CSV loader defined successfully.
Expected columns: 17



## 4. Data Validation and Cleaning

This section checks the integrity of the STP experimental dataset.

It identifies missing columns, duplicate experiments, invalid numeric values, inconsistent packet counts, and missing measurements. Raw measurements are preserved; invalid or missing values are not silently replaced with invented values.


In [6]:

# Cell 4: Data validation and cleaning functions

def validate_stp_data(df):
    """Validate an STP dataset without inventing missing measurements."""

    report = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "missing_values": df.isna().sum(),
        "duplicate_rows": int(df.duplicated().sum()),
        "invalid_numeric_values": {},
        "invalid_packet_counts": 0,
        "negative_metrics": {},
    }

    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            converted = pd.to_numeric(df[col], errors="coerce")
            report["invalid_numeric_values"][col] = int(
                (df[col].notna() & converted.isna()).sum()
            )

            report["negative_metrics"][col] = int(
                (converted < 0).sum()
            )

    if {"packets_sent", "packets_received"}.issubset(df.columns):
        sent = pd.to_numeric(df["packets_sent"], errors="coerce")
        received = pd.to_numeric(df["packets_received"], errors="coerce")

        invalid = (
            (sent < 0)
            | (received < 0)
            | (received > sent)
        )
        report["invalid_packet_counts"] = int(invalid.sum())

    print("DATA VALIDATION REPORT")
    print("=" * 40)
    print("Rows:", report["row_count"])
    print("Columns:", report["column_count"])
    print("Duplicate rows:", report["duplicate_rows"])

    print("\nMissing values by column:")
    display(report["missing_values"].to_frame("missing_count"))

    print("\nInvalid numeric entries:")
    display(pd.Series(report["invalid_numeric_values"], name="count").to_frame())

    print("\nNegative values by numeric column:")
    display(pd.Series(report["negative_metrics"], name="count").to_frame())

    print("\nRows with inconsistent packet counts:",
          report["invalid_packet_counts"])

    return report


def clean_stp_data(df):
    """
    Create a cleaned copy with numeric columns converted to numeric types.
    Does not impute missing values or delete experimental observations.
    """
    cleaned = df.copy()

    for col in NUMERIC_COLUMNS:
        if col in cleaned.columns:
            cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    return cleaned


print("Validation and cleaning functions defined successfully.")


Validation and cleaning functions defined successfully.



## 5. Performance Metric Calculations

This section calculates derived performance metrics from recorded experimental measurements.

Packet-loss percentage is calculated from packets sent and received. Convergence-time and downtime measurements remain separate. Missing raw measurements remain missing.


In [7]:

# Cell 5: Calculate derived metrics

def calculate_stp_metrics(df):
    """Return a copy of the dataset with derived metrics."""

    result = clean_stp_data(df)

    # Calculate packets lost only when sent and received are both available.
    if {"packets_sent", "packets_received"}.issubset(result.columns):
        sent = result["packets_sent"]
        received = result["packets_received"]

        valid_counts = (
            sent.notna()
            & received.notna()
            & (sent >= 0)
            & (received >= 0)
            & (received <= sent)
        )

        result["calculated_packets_lost"] = (
            sent - received
        ).where(valid_counts)

        # If packets_lost was independently recorded, preserve it.
        # Use the calculated value only in the derived column above.
        result["packet_loss_percent"] = (
            100 * (sent - received) / sent
        ).where(valid_counts & (sent > 0))

    # Create explicit units and names for downstream analysis.
    if "convergence_time_ms" in result.columns:
        result["convergence_time_s"] = (
            result["convergence_time_ms"] / 1000
        )

    if "downtime_ms" in result.columns:
        result["downtime_s"] = result["downtime_ms"] / 1000

    return result


print("Performance metric calculation function defined successfully.")


Performance metric calculation function defined successfully.



## 6. Summary Statistics by Topology

Summarize actual experimental measurements by topology. Mean and median convergence time are calculated only from available valid measurements. The number of valid observations is reported alongside each statistic.


In [8]:

# Cell 6: Summary statistics

def summarize_by_topology(df):
    """Summarize performance measurements by topology."""

    data = calculate_stp_metrics(df)

    if "topology" not in data.columns:
        raise ValueError("Dataset must contain a topology column.")

    metric_columns = [
        "convergence_time_ms",
        "downtime_ms",
        "packet_loss_percent",
        "blocked_ports",
        "forwarding_ports",
        "bpdu_count",
        "path_cost",
    ]

    available = [c for c in metric_columns if c in data.columns]

    if not available:
        raise ValueError("No supported performance metric columns found.")

    summary = data.groupby("topology", dropna=False)[available].agg(
        ["count", "mean", "median", "min", "max"]
    )

    return summary


print("Topology summary function defined successfully.")


Topology summary function defined successfully.



## 7. Failure Scenario Comparison

Compare experimental results by topology and failed link. Each row remains associated with its original experiment and trial.


In [9]:

# Cell 7: Failure scenario summary

def summarize_failures(df):
    """Summarize measured performance by topology and failed link."""

    data = calculate_stp_metrics(df)

    grouping = [
        c for c in ["topology", "failed_link"]
        if c in data.columns
    ]

    if not grouping:
        raise ValueError("No topology/failure columns available.")

    metrics = [
        c for c in [
            "convergence_time_ms",
            "downtime_ms",
            "packet_loss_percent",
            "blocked_ports",
            "forwarding_ports",
            "bpdu_count",
        ]
        if c in data.columns
    ]

    return data.groupby(grouping, dropna=False)[metrics].agg(
        ["count", "mean", "median"]
    )


print("Failure comparison function defined successfully.")


Failure comparison function defined successfully.



## 8. Export Processed Data and Summary Tables

Save processed data and summary tables for the visualization notebook and final report. Only actual imported experimental data should be exported as experimental results.


In [10]:

# Cell 8: Export functions

from pathlib import Path

OUTPUT_DIR = Path("results")
TABLE_DIR = OUTPUT_DIR / "tables"
PROCESSED_DIR = Path("data") / "processed"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def export_stp_analysis(df, prefix="stp"):
    """Export processed measurements and summary tables."""

    processed = calculate_stp_metrics(df)
    topology_summary = summarize_by_topology(df)
    failure_summary = summarize_failures(df)

    processed.to_csv(
        PROCESSED_DIR / f"{prefix}_processed.csv",
        index=False
    )

    topology_summary.to_csv(
        TABLE_DIR / f"{prefix}_topology_summary.csv"
    )

    failure_summary.to_csv(
        TABLE_DIR / f"{prefix}_failure_summary.csv"
    )

    print("Analysis exports saved:")
    print(PROCESSED_DIR / f"{prefix}_processed.csv")
    print(TABLE_DIR / f"{prefix}_topology_summary.csv")
    print(TABLE_DIR / f"{prefix}_failure_summary.csv")

    return processed, topology_summary, failure_summary


print("Export functions defined successfully.")


Export functions defined successfully.
